# MeChess 1/3: collect books, annotated games and explanations (CPU notebook)

**Accelerator: None.** Collecting and parsing text is CPU work and does not use your GPU quota. This notebook is only an adapter: it clones the repository and runs `chessme` commands.
Everything runs the same on a laptop: `python -m chessme books-learn --out data/books_learn`.

Gathers: 100+ public-domain books and periodicals, every annotated-game source (with glyphs `! ? !! ?? !? ?!` and evaluation symbols), chess Stack Exchange and Wikipedia prose, and Lichess opening names.

**Run:** Settings -> Internet On, Accelerator None; set `REPO_URL`; **Save Version -> Save & Run All (Commit)**. The command is resumable and skips finished work. Afterwards add this notebook's output as an input of
notebook 2. Terms: public-domain books only; annotated games and Stack Exchange / Wikipedia (CC BY-SA) keep their own licences: research and learning use, do not redistribute.

In [ ]:
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/<you>/MeChess.git"      # <- put your repository URL here

OUT = "/kaggle/working/learn" if os.path.exists("/kaggle") else "learn_local"

def sh(*args):
    """Run a command and stream its output; stop the notebook if it fails."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(args)}")

if not os.path.exists("chessme"):                      # Kaggle: fetch the repository (Internet must be on)
    if "<you>" in REPO_URL:
        raise SystemExit("Set REPO_URL to your repository first.")
    sh("git", "clone", "--depth", "1", REPO_URL, "MeChess")
    os.chdir("MeChess")
sh(sys.executable, "-m", "pip", "-q", "install", "python-chess", "numpy", "pyyaml", "requests")
CLI = [sys.executable, "-m", "chessme"]              # every step below is one `chessme` command: the same ones you run on a laptop

## Step 1. Preflight (about a minute): dependencies, disk and every download URL. Stops here if anything is missing.

In [ ]:
sh(*CLI, "books-preflight", "--out", OUT, "--min-free-gb", "4")

## Step 2. Collect everything (resumable; rerun the cell if the session was interrupted). `--slim` removes the raw downloads afterwards.

In [ ]:
sh(*CLI, "books-learn", "--out", OUT, "--slim", "--log", f"{OUT}/learn.log")

In [ ]:
print(pathlib.Path(OUT, "report.md").read_text())

## Optional: your own PDFs (private notebook only)
Upload PDFs as a *private* Kaggle dataset. A PDF with a text layer is read; a scan is reported as needing OCR. Never publish the outputs.

In [ ]:
if False:                 # set True for your own books
    sh(sys.executable, "-m", "pip", "-q", "install", "pypdf")
    sh(*CLI, "books-pdf", "/kaggle/input", "--out", f"{OUT}/private", "--log", f"{OUT}/pdf.log")